# 04 — Plan a fair time-based model test

To check if fraud predictions generalize to later transactions, we must not randomly mix the timeline. We will train on the earliest 70% of PaySim time steps, use the next 15% to choose settings, and keep the final 15% for a final evaluation.

The notebook scans only `step` and `isFraud` in batches on the first run, then saves a small local cache. Later runs reuse that cache unless the source CSV changes. It does not load the full transaction file into memory. We split by unique time steps so every transaction at the same step stays together.

In [ ]:
import json
from pathlib import Path

import pandas as pd

project_root = Path.cwd()
raw_data_dir = project_root / "data" / "raw"
csv_files = sorted(raw_data_dir.glob("*.csv"))
if not csv_files:
    raise FileNotFoundError(f"No PaySim CSV found in {raw_data_dir}")
data_path = csv_files[0]

processed_dir = project_root / "data" / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)
cache_path = processed_dir / "step_summary.csv"
cache_meta_path = processed_dir / "step_summary_source.json"
source_signature = {
    "file_name": data_path.name,
    "file_size_bytes": data_path.stat().st_size,
    "modified_time_ns": data_path.stat().st_mtime_ns,
}
cache_is_current = False
if cache_path.exists() and cache_meta_path.exists():
    try:
        cached_signature = json.loads(cache_meta_path.read_text(encoding="utf-8"))
        cache_is_current = cached_signature == source_signature
    except (OSError, json.JSONDecodeError):
        cache_is_current = False

if cache_is_current:
    step_summary = pd.read_csv(cache_path, index_col="step").sort_index()
    print(f"Loaded cached step summary for {data_path.name}")
else:
    from collections import Counter

    print("Building the step summary from the CSV (first run or source changed)...")
    transactions_by_step = Counter()
    frauds_by_step = Counter()
    for batch_number, chunk in enumerate(
        pd.read_csv(data_path, usecols=["step", "isFraud"], chunksize=500_000), start=1
    ):
        batch_summary = chunk.groupby("step")["isFraud"].agg(["size", "sum"])
        transactions_by_step.update(batch_summary["size"].to_dict())
        frauds_by_step.update(batch_summary["sum"].to_dict())
        print(f"Scanned batch {batch_number}")

    step_summary = pd.DataFrame({
        "transactions": pd.Series(transactions_by_step),
        "fraud_transactions": pd.Series(frauds_by_step),
    }).fillna(0).astype("int64").rename_axis("step").sort_index()
    step_summary.to_csv(cache_path, index_label="step")
    cache_meta_path.write_text(json.dumps(source_signature, indent=2), encoding="utf-8")
    print(f"Saved reusable summary to {cache_path}")
step_summary["step_number"] = step_summary.index
steps = step_summary.index.to_list()
if len(steps) < 3:
    raise ValueError("Not enough distinct time steps to create train, validation, and test periods.")

train_end_step = steps[int(len(steps) * 0.70) - 1]
validation_end_step = steps[int(len(steps) * 0.85) - 1]
step_summary["split"] = "test"
step_summary.loc[step_summary.index <= train_end_step, "split"] = "train"
step_summary.loc[
    (step_summary.index > train_end_step) & (step_summary.index <= validation_end_step),
    "split",
] = "validation"

split_summary = step_summary.groupby("split").agg(
    first_step=("step_number", "min"),
    last_step=("step_number", "max"),
    transactions=("transactions", "sum"),
    fraud_transactions=("fraud_transactions", "sum"),
)
split_summary["fraud_rate_percent"] = (
    split_summary["fraud_transactions"] / split_summary["transactions"] * 100
)
split_summary = split_summary.reindex(["train", "validation", "test"])[["first_step", "last_step", "transactions", "fraud_transactions", "fraud_rate_percent"]]

print(f"Number of time steps: {len(steps):,}")
print(f"Training period ends at step {train_end_step}")
print(f"Validation period ends at step {validation_end_step}")
display(split_summary)

## Why this split is fairer

Training uses only earlier steps. Validation comes after training and helps select a model or decision threshold. Test data is the latest period and stays untouched until final evaluation. If we randomly split rows, very similar transactions from the same time could land in both train and test, making the score look better than it would on genuinely later data.

The fraud rate may differ between periods. That is useful information about temporal robustness, not a reason to change the split after seeing the final test results.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
step_summary["transactions"].plot(ax=axes[0], color="#3976a8")
axes[0].set_title("Transactions by simulated time step")
axes[0].set_ylabel("Transaction count")
step_summary["fraud_transactions"].plot(ax=axes[1], color="#c44e52")
axes[1].set_title("Fraud-labeled transactions by time step")
axes[1].set_ylabel("Fraud count")
axes[1].set_xlabel("PaySim step")
for axis in axes:
    axis.axvline(train_end_step, color="#dd8452", linestyle="--", label="train / validation boundary")
    axis.axvline(validation_end_step, color="#8172b3", linestyle="--", label="validation / test boundary")
    axis.legend()
plt.tight_layout()
plt.show()

## Modeling note

These cutoffs are based on time steps, not exact row percentages, because transaction volumes vary by step. In the next notebook, we will use these saved cutoffs to build features and models without allowing later-period information into training.